<font color="#117A65" size='6px'><b>Video Object Counting</b></font>


<font color="#117A65" size='4px'><b>The Pipeline Architecture</b></font>

The system operates in a multi-stage pipeline designed for robustness against occlusion, motion blur, and class ambiguity.

<font color="#D35400" size='4px'><b>Stage I: Scene Understanding & Class Discovery (Frame 0)</b></font>
Before tracking begins, the system must determine what to count.

Automatic Mode (LLM/BLIP): The first frame is extracted and analyzed by a Vision-Language Model.

<font color="#884EA0"><i>LLM Mode:</i></font> Uses Async Parallel Processing to crop the image and identify complex objects (e.g., "specific brand logo").

<font color="#884EA0"><i>BLIP Mode:</i></font> Uses local captioning models to find standard nouns (e.g., "car", "person").

Refinement: The discovered nouns are cleaned via NLP to remove duplicates and abstract concepts.

Prompt Mode: The user manually specifies the target (e.g., prompt="red car"), skipping the discovery phase.

<font color="#D35400" size='4px'><b>Stage II: SAM 3 Tracker Initialization</b></font>
Once classes are defined, the SAM 3 Video Predictor is initialized.

Session Management: A unique "Inference Session" is created for the video.

Prompting Frame 0: The system inputs the target nouns (text prompts) into Frame 0. SAM 3 generates the initial Spatial Masks for every instance of those objects found in the first frame.

<font color="#D35400" size='4px'><b>Stage III: Temporal Propagation (The "Memory" Engine)</b></font>
Instead of re-detecting objects on every single frame (which causes flickering and ID switching), the system uses Propagation.

Memory Bank: SAM 3 builds a memory of the object's visual features.

Stream Tracking: As the video advances, the model "propels" the mask forward, adjusting for rotation, scaling, and deformation.

Occlusion Handling: If an object disappears behind a tree, the memory bank retains its ID. When it reappears, tracking resumes seamlessly.

<font color="#D35400" size='4px'><b>Stage IV: Trajectory Registration & Filtering</b></font>
Raw tracking data is noisy. The track_registry system cleans it in real-time.

Drift Check: It calculates the Euclidean distance of the object's center between frames. If an object "jumps" too far (impossible physics), it is flagged as <font color="#CB4335"><i>Lost/Drifted</i></font>.

Confidence Thresholding: Masks with low confidence scores (< 0.2) are discarded to prevent false positives.

Unique ID Assignment: Every validated object gets a persistent ID (e.g., Car_01, Person_42) linked to its entire history.

<font color="#D35400" size='4px'><b>Stage V: Visualization & Rendering</b></font>
The final step reconstructs the video with rich analytical overlays.

Dynamic Masking: Each class is assigned a unique, harmonious color.

Trajectory Drawing: The historical path of the object is drawn to show movement patterns.

Data Compression: The heavy pixel masks are compressed using RLE (Run-Length Encoding) and serialized into a .pkl file for efficient storage.

<font color="#2E86C1" size='6px'>Environment Initialization & Repository Setup</font>

In [ ]:
COLAB = True

In [ ]:
import os
import sys

if COLAB:
    # Clone Repository 
    if not os.path.exists("sam3-toolkit"):
        print("--- Cloning repository...")
        !git clone https://github.com/MrKGhasemi/sam3-toolkit.git
    else:
        print("--- Repository already exists.")

    # Auto-Detect Paths 
    repo_root = os.path.abspath("sam3-toolkit")
    sam3_install_path = None
    practical_path = None

    for root, dirs, files in os.walk(repo_root):
        if ("sam3" in dirs or "sam3_main" in root or root == repo_root):
            if sam3_install_path is None or len(root) < len(sam3_install_path):
                sam3_install_path = root

        if "object_counting.py" in files:
            practical_path = root

    if not sam3_install_path or not practical_path:
        raise FileNotFoundError(
            "     --- Critical paths (sam3 or practical) not found.")

    print(f"Found SAM 3 Library at: {sam3_install_path}")
    print(f"Found Project Code at: {practical_path}")

    # Install Dependencies 
    print("--- Installing Python packages...")
    !{sys.executable} -m pip install -q opencv-python matplotlib scikit-learn transformers spacy openai gdown mega.py huggingface_hub
    !{sys.executable} -m pip install -q einops decord pycocotools

    # Install SAM 3 
    print("--- Installing SAM 3 Library...")
    !{sys.executable} -m pip install -e "{sam3_install_path}"

    # Configure Working Directory 
    os.chdir(practical_path)
    if practical_path not in sys.path:
        sys.path.insert(0, practical_path)
    print(f"--- Working directory set to: {os.getcwd()}")

    # Download Model Weights
    weights_dir = "weights"
    weights_path = os.path.join(weights_dir, "sam3.pt")
    os.makedirs(weights_dir, exist_ok=True)

    # If the model is Private/Gated, you MUST provide a token.
    # Get it here: https://huggingface.co/settings/tokens
    HF_REPO_ID = "facebook/sam3" 
    HF_FILENAME = "sam3.pt"               
    HF_TOKEN = "HF_TOKEN"  # Replace with your actual token or set to None

    if not os.path.exists(weights_path):
        print(f"--- Starting download...")

        try:
            from huggingface_hub import hf_hub_download
            print("--- Connecting to Hugging Face Hub...")
            downloaded_file = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=HF_FILENAME,
                local_dir=weights_dir,
                token=HF_TOKEN,
                local_dir_use_symlinks=False  # Ensure we get the actual file, not a symlink
            )
            if os.path.basename(downloaded_file) != "sam3.pt":
                os.rename(downloaded_file, weights_path)

            print("--- Download attempt finished.")

        except Exception as e:
            print(f"      --- Error downloading: {e}")
            print("           Tip: If using Hugging Face private repo, ensure HF_TOKEN is set.")

    else:
        print("--- Weights file already exists.")

    # Verify File Integrity 
    if os.path.exists(weights_path):
        final_size = os.path.getsize(weights_path) / (1024 * 1024)
        print(f"--- Final File Size: {final_size:.2f} MB")
        if final_size < 2000:
            print(
                "      --- WARNING: File seems too small (<2GB). It might be corrupt or a placeholder.")
    else:
        print("      --- Error: File not found after download.")

    # Download SpaCy Model 
    print("--- Checking SpaCy model...")
    !{sys.executable} -m spacy download en_core_web_lg

    # Verify Import 
    print("\n--- Verifying imports...")
    repo_root = os.path.abspath("/sam3-toolkit")

    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
        print(f"--- Added '{repo_root}' to sys.path")

    practical_dir = os.path.join(repo_root, "practical")
    if practical_dir not in sys.path:
        sys.path.insert(0, practical_dir)
        print(f"--- Added '{practical_dir}' to sys.path")

    # Verify all dependencies
    print("\n--- Retrying imports...")
    try:
        import sam3
        print("--- 'sam3' library loaded!")
    except ImportError as e:
        print(f"      --- Failed to load sam3: {e}")
        print("       --- Attempting hard install fix...")
        !pip install "{repo_root}"
        import sam3

    try:
        import models
        print("--- 'models.py' loaded!")
    except ImportError as e:
        print(f"      --- Failed to load models: {e}")

    !pip uninstall numpy -y
    !{sys.executable} -m pip install "numpy<2.0"

    # Check GPU
    import torch
    print(f"--- GPU Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"--- GPU Name: {torch.cuda.get_device_name(0)}")

    if os.path.exists(practical_dir):
        os.chdir(practical_dir)
        print(f"--- Current Directory: {os.getcwd()}")


---

<font color="#117A65" size='6px'>Loading Core Modules & Dependencies</font>

<font color="#D68910" size='4px'>*object_counting:*</font> Contains the main pipeline for detecting and counting objects.

<font color="#D68910" size='4px'>*visualization:*</font> Handles the drawing of bounding boxes, masks, and legends.



In [ ]:
import os
import sys
root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root)

if os.path.basename(os.getcwd()) == "ipynb":
    os.chdir(root)

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
import object_counting
import torch
import torchvision
import visualization
import utils

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

In [ ]:
import numpy
print(numpy.__version__) 
# process is Not capable with numpy 2.0.0 or higher

```Python
# From configs.py 
LLM_CAPTION_MODEL_NAME = "gpt-5-chat"
SAM3_CONF_OBJECT_COUNTING_VIDEO_THRESHOLD = 0.2
```

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class MockArgs:
    def __init__(self, mode, output_dir, api_key=None, base_url=None):
        self.mode = mode
        self.output = output_dir
        self.api_key = api_key
        self.base_url = base_url


output_directory = "notebook_outputs"
os.makedirs(output_directory, exist_ok=True)

args = MockArgs(mode="blip", output_dir=output_directory)

# for LLM mode:
# args = MockArgs(
#     mode="llm",
#     output_dir=output_directory,
#     api_key="api_key",
#     base_url="base_url"
# )

---

<font color="#D35400" size='6px'>Video Execution: Prompt-Based Tracking</font>

Prompt-Based Counting Mode. By setting <font color="#D35400" size='4px'>task_type="prompt"</font> and providing a specific class (e.g., "person"), instruct the system to track only that object throughout the video.

The function <font color="#D35400"><i>count_objects_video</i></font> orchestrates a sophisticated tracking pipeline:

<font color="#D35400" size='4px'><i>Frame 0 Initialization:</i></font> It starts by identifying the target object on the very first frame. The system adds a text prompt (e.g., "person") to the <font color="#D35400"><i>SAM 3 Video Predictor</i></font>, which segments all initial instances of that object.

<font color="#D35400" size='4px'><i>Temporal Propagation:</i></font> Instead of re-detecting objects on every frame (which causes flickering), it uses the <font color="#D35400"><i>propagate_in_video</i></font> stream request. This leverages the SAM 3 memory bank to "remember" the object's appearance and track its movement seamlessly across time.

<font color="#D35400" size='4px'><i>Trajectory Registration:</i></font> As the video plays, the system maintains a <font color="#D35400"><i>track_registry</i></font>. It assigns a unique ID to each object and records its path (trajectory), filtering out noise or weak detections based on the <font color="#D35400"><i>SAM3_CONF_OBJECT_COUNTING_VIDEO_THRESHOLD</i></font>.

```python
# Video Propagation Logic
with torch.inference_mode():
    stream = predictor.handle_stream_request(request=dict(
        type="propagate_in_video", session_id=session_id
    ))
    
    for r in tqdm(stream):
        # r['outputs'] contains masks for the current frame
        masks_tensor = out_data.get("out_binary_masks")
        # ... logic to filter and save ...
```        

In [ ]:
VIDEO_PATH = "videos/walking.mp4"

In [ ]:
output_information = object_counting.count_objects_video(VIDEO_PATH, args, task_type="prompt", prompt_value="person")

<font color="#8E44AD" size='6px'>Video Visualization & Rendering</font>

The function <font color="#8E44AD"><i>counting_visualizations</i></font> takes the raw tracking data (trajectories, masks, class IDs) and reconstructs the video with rich annotations.

The rendering process involves three key layers:

<font color="#8E44AD" size='4px'><i>Frame Reconstruction:</i></font> It re-opens the original video file and processes it frame-by-frame. For each frame, it retrieves the corresponding masks and bounding boxes from the <font color="#8E44AD"><i>inference_results</i></font> dictionary.

<font color="#8E44AD" size='4px'><i>Dynamic Overlay:</i></font> It applies a translucent color mask to each object (using the harmonious color palette) and draws a bounding box with the unique Object ID (e.g., "Person_1", "Car_5"). This visualizes not just where the object is, but which specific instance it is.

<font color="#8E44AD" size='4px'><i>Drift & Count Display:</i></font> It calculates the center of the bounding box to draw the motion path (trajectory) and updates the on-screen counters in real-time, finally writing the processed frame to a new .mp4 file using <font color="#8E44AD"><i>cv2.VideoWriter</i></font>.

In [ ]:
visualization.counting_visualizations(output_information)

<font color="#2874A6" size='6px'>Universal Data Serialization</font>

This cell executes the Data Compression & Archiving step. Regardless of the specific task (Counting, Segmentation, or Redaction), the results—often containing thousands of high-resolution binary masks—are massive. This function efficiently packs them for storage.

The function <font color="#2874A6"><i>utils.compress_and_save_output_information</i></font> applies a rigorous optimization pipeline:

<font color="#2874A6" size='4px'><i>RLE Compression:</i></font> Instead of saving dense boolean arrays (which consume huge disk space), the function converts every segmentation mask into <font color="#2874A6"><i>Run-Length Encoding (RLE)</i></font> using <font color="#2874A6"><i>pycocotools</i></font>. This compresses the data by storing counts of consecutive pixels rather than the pixels themselves.

<font color="#2874A6" size='4px'><i>Memory Optimization:</i></font> Before encoding, the binary masks are converted to <font color="#2874A6"><i>Fortran-contiguous arrays</i></font> via <font color="#2874A6"><i>np.asfortranarray</i></font>. This specific memory layout is required for high-speed C-based encoding operations, ensuring the compression process doesn't bottleneck the pipeline.

<font color="#2874A6" size='4px'><i>Structured Pickling:</i></font> The simplified, compressed dictionary is serialized using Python's <font color="#2874A6"><i>pickle</i></font> module. This preserves the complex nested structure (frames -> objects -> metadata) in a single file that can be instantly reloaded for analysis later.

In [ ]:
save_path = os.path.join(output_directory, "object_counting_output_walking_compressed.pkl")

utils.compress_and_save_output_information(output_information, save_path)

___

<font color="#3c983c" size='6px'>Video Execution: LLM-Enhanced Automatic Counting</font>

This cell runs the Automatic Counting Mode powered by a Multimodal LLM. By setting <font color="#3c983c" size='4px'>args.mode='llm'</font> and <font color="#3c983c" size='4px'>task_type="automatic"</font>, the system autonomously discovers and tracks all relevant objects in the video without user prompts.

This function combines high-level reasoning with low-level tracking:

<font color="#CB4335" size='4px'><i>Scene Understanding (Frame 0):</i></font> The system extracts the first frame and sends it to the <font color="#CB4335"><i>get_classes_llm</i></font> function. The LLM analyzes the scene in parallel (using asyncio) to generate a comprehensive list of "target nouns" present in the video (e.g., "ball", "player", "goal post").

<font color="#CB4335" size='4px'><i>Multi-Class Propagation:</i></font> The system iterates through each discovered class independently. For every class, it initializes a new tracking session with SAM 3, propagating that specific object type through the entire video duration to ensure precise, class-specific trajectories.

<font color="#CB4335" size='4px'><i>Result Aggregation:</i></font> Finally, the tracks from all different classes are merged into a single timeline. The system resolves overlaps and compiles a complete history of every object's movement for the final visualization.

In [ ]:
VIDEO_PATH = "videos/football.mp4"

In [ ]:
# output_information = object_counting.count_objects_video(VIDEO_PATH, args, task_type="automatic")

In [ ]:
# visualization.save_video_counting_visualizations(output_information)

In [ ]:
# for LLM mode:
args = MockArgs(
    mode="llm",
    output_dir=output_directory,
    api_key="api_key",
    base_url="base_url"
)

os.makedirs(args.output, exist_ok=True)

In [ ]:
output_information2 = object_counting.count_objects_video(VIDEO_PATH, args, task_type="automatic")

In [ ]:
visualization.counting_visualizations(output_information2)

In [ ]:
save_path = os.path.join(output_directory, "object_counting_output_football_compressed.pkl")

utils.compress_and_save_output_information(output_information2, save_path)